
# JWST Public Data Analysis from MAST (No Local Files Needed)

This notebook is a **MAST-first**, analysis-oriented workflow for **public JWST telescope data**.

It is designed for your exact situation:

- you **do not have local FITS files**
- you want the notebook to **download public data from the web**
- you want **analysis**, not just a pretty image

## What this notebook does

1. Verifies internet reachability to MAST
2. Searches **public JWST observations** by target name
3. Retrieves a **small, stable set of image products** from MAST
4. Downloads those products into a local notebook folder
5. Loads FITS science image extensions
6. Cleans the data and estimates backgrounds
7. Measures simple source properties:
   - brightest source location
   - centroid
   - aperture photometry
   - radial profile
   - row/column detector diagnostics
8. Optionally aligns filters and builds an RGB quicklook
9. Exports tables and analysis summaries

## Why this version is more reliable

This notebook intentionally avoids a few common failure modes:

- it defaults to **MAST mode**
- it limits the number of observations expanded at once
- it uses **public-only** filtering
- it prefers **high-level calibrated image products**
- it enables **cloud-backed downloads** when available
- it includes a **connectivity test** so network problems are obvious

## If downloads still fail

If the connectivity test cell cannot reach `https://mast.stsci.edu`, then the problem is usually the notebook environment's internet access, not the astronomy code.


In [ ]:

# Run this once in a fresh environment.
%pip -q install -U astroquery astropy photutils reproject scikit-image pillow matplotlib scipy pandas requests boto3 botocore


## Imports and version check

In [ ]:

from pathlib import Path
import warnings
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from PIL import Image
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm, make_lupton_rgb
from astropy.wcs import WCS
from astropy.table import vstack, unique, Table

from astroquery.mast import Observations
from scipy.ndimage import median_filter
from scipy.ndimage import center_of_mass
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from reproject import reproject_interp

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print("astroquery version check:")
try:
    import astroquery, astropy, photutils
    print("  astroquery:", astroquery.__version__)
    print("  astropy   :", astropy.__version__)
    print("  photutils :", photutils.__version__)
except Exception as e:
    print("Version check warning:", e)

plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["image.origin"] = "lower"


## Configuration

In [ ]:

# Default mode: download public JWST data from MAST
DATA_MODE = "mast"   # keep this as "mast" unless you later add local FITS files

# Search target
MAST_TARGET = "M16"           # examples: "M16", "NGC 346", "SMACS 0723"
MAST_RADIUS_DEG = 0.03
MAST_MAX_OBS = 3              # small on purpose; helps avoid timeouts
PRODUCT_LIMIT = 6             # maximum number of FITS files to download

# Optional narrowing
JWST_ONLY = True
PUBLIC_ONLY = True
IMAGE_ONLY = True

# Prefer high-level calibrated image products
PREFERRED_SUBGROUPS = ["I2D", "DRZ", "DRC", "CAL", "SCI"]
PREFERRED_FILENAME_KEYWORDS = ["i2d", "drz", "drc", "cal", "rate", "sci"]

# Local folders created by the notebook
DOWNLOAD_DIR = Path("./mast_downloads")
WORK_DIR = Path("./work")
OUTPUT_DIR = Path("./output")
for d in [DOWNLOAD_DIR, WORK_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Analysis / display parameters
BACKGROUND_SIGMA = 3.0
BACKGROUND_MAXITERS = 10
BACKGROUND_PERCENTILE_LOW = 1.0
BACKGROUND_PERCENTILE_HIGH = 99.7
STRETCH = "asinh"  # "asinh", "sqrt", "log", "linear"

# Optional RGB quicklook
RGB_ENABLED = True
USE_LUPTON_RGB = False

# Optional cloud-backed downloads for public data
ENABLE_CLOUD_DOWNLOAD = True

# For direct product retries
HTTP_TIMEOUT = 60
CACHE_DOWNLOADS = True


## Step 1 — Connectivity test

In [ ]:

def check_url(url, timeout=20):
    try:
        r = requests.get(url, timeout=timeout)
        return {"url": url, "ok": True, "status_code": r.status_code, "detail": "reachable"}
    except Exception as e:
        return {"url": url, "ok": False, "status_code": None, "detail": str(e)}

checks = [
    check_url("https://mast.stsci.edu"),
    check_url("https://archive.stsci.edu"),
]

connectivity_df = pd.DataFrame(checks)
display(connectivity_df)

if not connectivity_df["ok"].all():
    raise RuntimeError(
        "Network test failed. This notebook environment could not reach one or more MAST/STScI endpoints. "
        "If this happens in Colab, Kaggle, or a hosted notebook, outbound internet may be blocked or unstable."
    )
else:
    print("Connectivity looks OK.")


## Step 2 — Enable cloud-backed public downloads when available

In [ ]:

if ENABLE_CLOUD_DOWNLOAD:
    try:
        Observations.enable_cloud_dataset()
        print("Cloud-backed public download enabled.")
    except Exception as e:
        print("Could not enable cloud-backed downloads:", e)
        print("Falling back to standard MAST downloads.")


## Helper functions

In [ ]:

def list_fits_files(folder: Path):
    if not folder.exists():
        return []
    files = []
    for pat in ["*.fits", "*.fit", "*.fts", "*.fits.gz"]:
        files.extend(folder.rglob(pat))
    return sorted(set(files))


def choose_science_hdu(hdul):
    candidates = []
    for idx, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None or not isinstance(data, np.ndarray) or data.ndim < 2:
            continue
        header = hdu.header
        name = (getattr(hdu, "name", "") or "").upper()
        score = 0
        if name == "SCI":
            score += 100
        if data.ndim == 2:
            score += 25
        if idx == 0:
            score += 10
        if "WCSAXES" in header or "CTYPE1" in header:
            score += 30
        if data.shape[-2] > 128 and data.shape[-1] > 128:
            score += 20
        candidates.append((score, idx, hdu))
    if not candidates:
        return None, None, None
    candidates.sort(reverse=True, key=lambda x: x[0])
    _, idx, hdu = candidates[0]
    return idx, np.array(hdu.data), hdu.header


def robust_clean(data):
    x = np.array(data, dtype=np.float32)
    bad = ~np.isfinite(x)
    if bad.any():
        good = ~bad
        fill = np.nanmedian(x[good]) if good.any() else 0.0
        x[bad] = fill
    return x


def background_subtract(data, sigma=BACKGROUND_SIGMA, maxiters=BACKGROUND_MAXITERS):
    mean, median, std = sigma_clipped_stats(data, sigma=sigma, maxiters=maxiters)
    return data - median, {"mean": float(mean), "median": float(median), "std": float(std)}


def remove_hot_pixels(data, size=3):
    med = median_filter(data, size=size)
    resid = data - med
    _, _, std = sigma_clipped_stats(resid, sigma=3.0)
    mask = resid > 8 * std
    out = data.copy()
    out[mask] = med[mask]
    return out


def normalize_channel(data, low=BACKGROUND_PERCENTILE_LOW, high=BACKGROUND_PERCENTILE_HIGH, stretch=STRETCH):
    x = np.array(data, dtype=np.float32)
    lo = np.nanpercentile(x, low)
    hi = np.nanpercentile(x, high)
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    if stretch == "sqrt":
        x = np.sqrt(x)
    elif stretch == "log":
        x = np.log1p(1000 * x) / np.log1p(1000)
    elif stretch == "asinh":
        a = 8.0
        x = np.arcsinh(a * x) / np.arcsinh(a)
    elif stretch != "linear":
        raise ValueError(f"Unknown stretch: {stretch}")
    return np.clip(x, 0, 1)


def detect_filter_name(header, path=None):
    keys = ["FILTER", "PUPIL", "FILTNAM1", "FILTNAM2", "FILTER1", "FILTER2"]
    vals = []
    for k in keys:
        if k in header and str(header[k]).strip():
            vals.append(str(header[k]).strip().lower())
    if path is not None:
        stem = Path(path).stem.lower()
        vals.extend(re.findall(r"f\d{3}[wmn]", stem))
    vals = [v for v in vals if v not in {"clear", "none", "nan"}]
    if vals:
        for v in vals:
            if re.match(r"f\d{3}[wmn]", v):
                return v
        return vals[0]
    return Path(path).stem.lower() if path is not None else "unknown"


def wavelength_key(name):
    m = re.search(r"f(\d{3})([wmn])", str(name).lower())
    return int(m.group(1)) if m else 9999


def auto_assign_rgb(filter_names):
    ordered = sorted(filter_names, key=wavelength_key)
    n = len(ordered)
    if n == 1:
        return {"R": ordered[0], "G": ordered[0], "B": ordered[0]}
    if n == 2:
        return {"R": ordered[1], "G": ordered[0], "B": ordered[0]}
    if n == 3:
        return {"B": ordered[0], "G": ordered[1], "R": ordered[2]}
    return {"B": ordered[0], "G": ordered[n // 2], "R": ordered[-1]}


def radial_profile(data, center=None, binsize=1):
    y, x = np.indices(data.shape)
    if center is None:
        center = (data.shape[1] / 2, data.shape[0] / 2)
    cx, cy = center
    r = np.sqrt((x - cx)**2 + (y - cy)**2)
    rbin = (r / binsize).astype(int)
    tbin = np.bincount(rbin.ravel(), data.ravel())
    nr = np.bincount(rbin.ravel())
    prof = np.divide(tbin, nr, out=np.zeros_like(tbin, dtype=float), where=nr > 0)
    radius = np.arange(len(prof)) * binsize
    return radius, prof


def brightest_pixel_xy(data):
    iy, ix = np.unravel_index(np.nanargmax(data), data.shape)
    return int(ix), int(iy)


def centroid_near_peak(data, peak_xy, half_size=12):
    x0, y0 = peak_xy
    y1 = max(0, y0 - half_size)
    y2 = min(data.shape[0], y0 + half_size + 1)
    x1 = max(0, x0 - half_size)
    x2 = min(data.shape[1], x0 + half_size + 1)
    cut = data[y1:y2, x1:x2]
    cut_shift = cut - np.nanmin(cut)
    if np.allclose(cut_shift.sum(), 0):
        return float(x0), float(y0)
    cy, cx = center_of_mass(cut_shift)
    return float(x1 + cx), float(y1 + cy)


def aperture_photometry_with_background(data, x, y, r=6.0, r_in=8.0, r_out=12.0):
    pos = [(x, y)]
    aper = CircularAperture(pos, r=r)
    ann = CircularAnnulus(pos, r_in=r_in, r_out=r_out)

    aper_tbl = aperture_photometry(data, aper)
    ann_tbl = aperture_photometry(data, ann)

    aper_sum = float(aper_tbl["aperture_sum"][0])
    ann_sum = float(ann_tbl["aperture_sum"][0])

    aper_area = float(aper.area)
    ann_area = float(ann.area)
    bkg_mean = ann_sum / ann_area if ann_area > 0 else 0.0
    net_flux = aper_sum - bkg_mean * aper_area

    return {
        "x": float(x),
        "y": float(y),
        "aperture_r": float(r),
        "annulus_r_in": float(r_in),
        "annulus_r_out": float(r_out),
        "aperture_sum": aper_sum,
        "annulus_sum": ann_sum,
        "background_mean_per_pix": bkg_mean,
        "net_flux": net_flux,
    }


def safe_display_image(data, title=None, cmap="gray", figsize=(8, 8)):
    plt.figure(figsize=figsize)
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    plt.imshow(data, cmap=cmap, norm=norm)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


## Step 3 — Search public JWST observations

In [ ]:

def search_public_jwst_observations(target=MAST_TARGET, radius_deg=MAST_RADIUS_DEG, max_obs=MAST_MAX_OBS):
    print(f"Searching MAST around target={target!r}, radius={radius_deg} deg ...")
    obs = Observations.query_object(target, radius=f"{radius_deg} deg")
    if len(obs) == 0:
        raise RuntimeError("No observations found at that target/radius.")

    df = obs.to_pandas()

    if JWST_ONLY and "obs_collection" in df.columns:
        df = df[df["obs_collection"].astype(str).str.upper() == "JWST"]

    if PUBLIC_ONLY and "dataRights" in df.columns:
        rights = df["dataRights"].astype(str).str.upper()
        df = df[(rights == "PUBLIC") | (rights == "")]

    if IMAGE_ONLY and "dataproduct_type" in df.columns:
        dtype = df["dataproduct_type"].astype(str).str.lower()
        df = df[dtype.str.contains("image", na=False) | dtype.eq("")]

    if len(df) == 0:
        raise RuntimeError("Observations were found initially, but none remained after JWST/public/image filtering.")

    sort_cols = [c for c in ["t_min", "proposal_id", "obs_id"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, ascending=False)

    # keep just a few observations to avoid huge product lists and timeouts
    key_col = "obsid" if "obsid" in df.columns else df.columns[0]
    df = df.drop_duplicates(subset=[key_col]).head(max_obs)

    selected = obs[np.isin(obs["obsid"], df["obsid"].tolist())]

    print(f"Selected {len(selected)} observation(s) after filtering.")
    return selected, df

obs_rows, obs_df = search_public_jwst_observations()
display(obs_df.head(10))


## Step 4 — Retrieve product lists one observation at a time

In [ ]:

def gather_products_for_observations(obs_rows):
    tables = []
    for i, row in enumerate(obs_rows):
        print(f"Fetching product list for observation {i+1}/{len(obs_rows)} ...")
        prod = Observations.get_product_list(row)
        if len(prod):
            tables.append(prod)
    if not tables:
        raise RuntimeError("No data products were returned for the selected observations.")
    merged = unique(vstack(tables), keys="dataURI" if "dataURI" in tables[0].colnames else "productFilename")
    return merged

products = gather_products_for_observations(obs_rows)
print(f"Raw product count: {len(products)}")
products[:5]


## Step 5 — Filter to a small, useful FITS product set

In [ ]:

def filter_products_for_download(products, product_limit=PRODUCT_LIMIT):
    filtered = products

    # Use astroquery's server-side/metadata-aware filtering when possible.
    try:
        filtered = Observations.filter_products(
            filtered,
            extension=["fits", "fits.gz"],
            productType=["SCIENCE", "PRODUCT", "!AUXILIARY"]
        )
    except Exception as e:
        print("filter_products warning:", e)

    pdf = filtered.to_pandas()

    if len(pdf) == 0:
        raise RuntimeError("No products left after basic FITS/science filtering.")

    for col in ["productFilename", "productSubGroupDescription", "dataRights", "calib_level", "productType"]:
        if col not in pdf.columns:
            pdf[col] = ""

    pdf["productFilename_l"] = pdf["productFilename"].astype(str).str.lower()
    pdf["subgroup_l"] = pdf["productSubGroupDescription"].astype(str).str.upper()
    pdf["rights_u"] = pdf["dataRights"].astype(str).str.upper()
    pdf["ptype_u"] = pdf["productType"].astype(str).str.upper()

    fits_mask = pdf["productFilename_l"].str.endswith(".fits") | pdf["productFilename_l"].str.endswith(".fits.gz")
    public_mask = (pdf["rights_u"] == "PUBLIC") | (pdf["rights_u"] == "")
    subgroup_score = pdf["subgroup_l"].isin(PREFERRED_SUBGROUPS).astype(int)
    keyword_score = pdf["productFilename_l"].apply(lambda s: int(any(k in s for k in PREFERRED_FILENAME_KEYWORDS)))
    type_score = pdf["ptype_u"].isin(["SCIENCE", "PRODUCT"]).astype(int)

    pdf = pdf[fits_mask & public_mask].copy()
    if len(pdf) == 0:
        raise RuntimeError("No public FITS files remained after filtering.")

    # Higher scores first, then smaller-ish curated subset.
    pdf["rank_score"] = 10 * subgroup_score.loc[pdf.index] + 5 * keyword_score.loc[pdf.index] + 2 * type_score.loc[pdf.index]
    if "size" in pdf.columns:
        pdf["size_num"] = pd.to_numeric(pdf["size"], errors="coerce")
    else:
        pdf["size_num"] = np.nan

    pdf = pdf.sort_values(["rank_score", "size_num"], ascending=[False, True])

    # Prefer unique filenames
    pdf = pdf.drop_duplicates(subset=["productFilename"]).head(product_limit).reset_index(drop=True)

    # Rebuild astropy table subset
    keep_names = set(pdf["productFilename"].tolist())
    mask = np.array([str(row["productFilename"]) in keep_names for row in filtered], dtype=bool)
    final = filtered[mask]

    # Preserve the chosen order using pandas view for display
    return final, pdf

download_products_table, download_products_df = filter_products_for_download(products)
print(f"Products selected for download: {len(download_products_table)}")
display(download_products_df[[
    c for c in ["obsID", "obs_id", "productFilename", "productSubGroupDescription", "productType", "dataRights", "size", "rank_score"]
    if c in download_products_df.columns
]])


## Step 6 — Download the selected products

In [ ]:

def download_selected_products(products_table, download_dir=DOWNLOAD_DIR, cache=CACHE_DOWNLOADS):
    download_dir.mkdir(parents=True, exist_ok=True)

    manifest_rows = []
    for row in products_table:
        fname = str(row["productFilename"]) if "productFilename" in row.colnames else "unknown.fits"
        uri = str(row["dataURI"]) if "dataURI" in row.colnames else None
        target_path = download_dir / fname

        if uri is None:
            manifest_rows.append({"productFilename": fname, "status": "ERROR", "message": "Missing dataURI", "local_path": None})
            continue

        try:
            status, msg, url = Observations.download_file(
                uri,
                local_path=str(target_path),
                cache=cache,
                verbose=True
            )
            manifest_rows.append({
                "productFilename": fname,
                "status": status,
                "message": msg,
                "url": url,
                "local_path": str(target_path) if target_path.exists() else None,
            })
        except Exception as e:
            manifest_rows.append({
                "productFilename": fname,
                "status": "ERROR",
                "message": str(e),
                "url": None,
                "local_path": None,
            })

    return pd.DataFrame(manifest_rows)

download_manifest = download_selected_products(download_products_table)
display(download_manifest)

ok_files = [Path(p) for p in download_manifest["local_path"].dropna().tolist() if Path(p).exists()]
print(f"Downloaded FITS files: {len(ok_files)}")

if len(ok_files) == 0:
    raise RuntimeError(
        "No FITS files were downloaded. Check the connectivity test, the manifest above, "
        "and whether your environment allows outbound internet access."
    )


## Step 7 — Inspect downloaded FITS files

In [ ]:

records = []
for path in ok_files:
    try:
        with fits.open(path) as hdul:
            idx, data, header = choose_science_hdu(hdul)
            if data is None:
                print(f"Skipping {path.name}: no usable 2D science image found")
                continue
            filt = detect_filter_name(header, path)
            records.append({
                "path": str(path),
                "filename": path.name,
                "hdu_index": idx,
                "filter_name": filt,
                "shape": tuple(data.shape),
                "instrument": str(header.get("INSTRUME", "")),
                "detector": str(header.get("DETECTOR", "")),
                "telescope": str(header.get("TELESCOP", "")),
                "target": str(header.get("TARGNAME", "")),
            })
    except Exception as e:
        print(f"Skipping {path.name}: {e}")

catalog = pd.DataFrame(records)
display(catalog)

if len(catalog) == 0:
    raise RuntimeError("Downloaded files exist, but none exposed a usable 2D image science extension.")


## Step 8 — Preview one image per filter

In [ ]:

def load_record_image(rec):
    with fits.open(rec["path"]) as hdul:
        data = robust_clean(hdul[rec["hdu_index"]].data)
    return data

unique_filters = catalog["filter_name"].dropna().unique().tolist()
preview_rows = []
for filt in unique_filters:
    rec = catalog[catalog["filter_name"] == filt].iloc[0]
    data = load_record_image(rec)
    data, stats = background_subtract(data)
    data = remove_hot_pixels(data)
    preview_rows.append((filt, data, stats))

n = len(preview_rows)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]

for ax, (filt, data, stats) in zip(axes, preview_rows):
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    ax.imshow(data, norm=norm, cmap="gray")
    ax.set_title(f"{filt}\nmedian={stats['median']:.3g}, std={stats['std']:.3g}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## Step 9 — Choose one representative image per filter

In [ ]:

catalog["n_pix"] = catalog["shape"].apply(lambda s: int(np.prod(s[-2:])))
best_per_filter = (
    catalog.sort_values("n_pix", ascending=False)
           .drop_duplicates("filter_name")
           .reset_index(drop=True)
)

display(best_per_filter[["filter_name", "filename", "path", "hdu_index", "shape", "n_pix"]])


## Step 10 — Load cleaned images and basic per-filter metrics

In [ ]:

loaded = {}
metrics_rows = []

for _, row in best_per_filter.iterrows():
    filt = row["filter_name"]
    with fits.open(row["path"]) as hdul:
        data = robust_clean(hdul[int(row["hdu_index"])].data)
        header = hdul[int(row["hdu_index"])].header

    clean, stats = background_subtract(data)
    clean = remove_hot_pixels(clean)

    peak_xy = brightest_pixel_xy(clean)
    cx, cy = centroid_near_peak(clean, peak_xy, half_size=12)
    phot = aperture_photometry_with_background(clean, cx, cy, r=6, r_in=8, r_out=12)

    loaded[filt] = {
        "data": clean,
        "header": header,
        "wcs": WCS(header),
        "stats": stats,
        "path": row["path"],
        "peak_xy": peak_xy,
        "centroid_xy": (cx, cy),
        "photometry": phot,
    }

    metrics_rows.append({
        "filter_name": filt,
        "path": row["path"],
        "mean": stats["mean"],
        "median": stats["median"],
        "std": stats["std"],
        "peak_x": peak_xy[0],
        "peak_y": peak_xy[1],
        "centroid_x": cx,
        "centroid_y": cy,
        "net_flux": phot["net_flux"],
        "background_mean_per_pix": phot["background_mean_per_pix"],
    })

metrics_df = pd.DataFrame(metrics_rows).sort_values("filter_name", key=lambda s: s.map(wavelength_key))
display(metrics_df)


## Step 11 — Inspect source location and apertures

In [ ]:

for filt, item in loaded.items():
    data = item["data"]
    x_peak, y_peak = item["peak_xy"]
    cx, cy = item["centroid_xy"]

    plt.figure(figsize=(7, 7))
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    plt.imshow(data, cmap="gray", norm=norm)
    plt.scatter([x_peak], [y_peak], s=80, marker="x", label="brightest pixel")
    plt.scatter([cx], [cy], s=80, marker="+", label="centroid")
    circ = plt.Circle((cx, cy), 6, fill=False)
    ann1 = plt.Circle((cx, cy), 8, fill=False, linestyle="--")
    ann2 = plt.Circle((cx, cy), 12, fill=False, linestyle="--")
    plt.gca().add_patch(circ)
    plt.gca().add_patch(ann1)
    plt.gca().add_patch(ann2)
    plt.title(f"Source diagnostics — {filt}")
    plt.legend()
    plt.axis("off")
    plt.show()


## Step 12 — Radial profiles

In [ ]:

for filt, item in loaded.items():
    data = item["data"]
    cx, cy = item["centroid_xy"]
    radius, profile = radial_profile(data, center=(cx, cy), binsize=1)

    plt.figure(figsize=(8, 4))
    plt.plot(radius[:200], profile[:200])
    plt.xlabel("Radius [pixels]")
    plt.ylabel("Mean intensity")
    plt.title(f"Radial profile — {filt}")
    plt.grid(True, alpha=0.3)
    plt.show()


## Step 13 — Detector-style row/column diagnostics

In [ ]:

for filt, item in loaded.items():
    data = item["data"]
    row_med = np.nanmedian(data, axis=1)
    col_med = np.nanmedian(data, axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(row_med)
    axes[0].set_title(f"Row median profile — {filt}")
    axes[0].set_xlabel("Row index")
    axes[0].set_ylabel("Median signal")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(col_med)
    axes[1].set_title(f"Column median profile — {filt}")
    axes[1].set_xlabel("Column index")
    axes[1].set_ylabel("Median signal")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Step 14 — Optional WCS alignment and RGB quicklook

In [ ]:

aligned = {}
if len(loaded) == 0:
    raise RuntimeError("No loaded filters available.")

selected_filters = sorted(list(loaded.keys()), key=wavelength_key)
reference_filter = selected_filters[0]
ref = loaded[reference_filter]
ref_header = ref["header"]
ref_shape = ref["data"].shape

for filt, item in loaded.items():
    if filt == reference_filter:
        aligned[filt] = item["data"]
    else:
        try:
            reprojected, footprint = reproject_interp((item["data"], item["wcs"]), ref_header, shape_out=ref_shape)
            aligned[filt] = robust_clean(reprojected)
            print(f"Aligned {filt} -> {reference_filter}")
        except Exception as e:
            print(f"Could not align {filt}: {e}")

print("Aligned filters:", list(aligned.keys()))


In [ ]:

if RGB_ENABLED and len(aligned) >= 1:
    rgb_assignment = auto_assign_rgb(list(aligned.keys()))
    print("RGB assignment:", rgb_assignment)

    r = normalize_channel(aligned[rgb_assignment["R"]])
    g = normalize_channel(aligned[rgb_assignment["G"]])
    b = normalize_channel(aligned[rgb_assignment["B"]])

    rgb_float = np.dstack([r, g, b])

    plt.figure(figsize=(10, 10))
    plt.imshow(rgb_float)
    plt.title("RGB quicklook")
    plt.axis("off")
    plt.show()

    if USE_LUPTON_RGB:
        try:
            lupton = make_lupton_rgb(
                aligned[rgb_assignment["R"]],
                aligned[rgb_assignment["G"]],
                aligned[rgb_assignment["B"]],
                stretch=5, Q=8
            )
            plt.figure(figsize=(10, 10))
            plt.imshow(lupton)
            plt.title("Lupton RGB quicklook")
            plt.axis("off")
            plt.show()
        except Exception as e:
            print("Lupton RGB failed:", e)


## Step 15 — Export analysis tables

In [ ]:

metrics_csv = OUTPUT_DIR / "jwst_filter_metrics.csv"
manifest_csv = OUTPUT_DIR / "jwst_download_manifest.csv"
catalog_csv = OUTPUT_DIR / "jwst_catalog.csv"
summary_json = OUTPUT_DIR / "jwst_summary.json"

metrics_df.to_csv(metrics_csv, index=False)
download_manifest.to_csv(manifest_csv, index=False)
catalog.to_csv(catalog_csv, index=False)

summary = {
    "target": MAST_TARGET,
    "radius_deg": MAST_RADIUS_DEG,
    "downloaded_files": [str(p) for p in ok_files],
    "filters": sorted(list(loaded.keys()), key=wavelength_key),
    "reference_filter": reference_filter if "reference_filter" in globals() else None,
}
with open(summary_json, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print(" -", metrics_csv)
print(" -", manifest_csv)
print(" -", catalog_csv)
print(" -", summary_json)



## Troubleshooting notes

### 1) The connectivity test fails
Your notebook runtime cannot reach MAST/STScI endpoints. That is usually an environment/network restriction, not a JWST code issue.

### 2) Search works but no files download
Try:
- a different target, such as `NGC 346`, `M16`, or `SMACS 0723`
- lowering `MAST_MAX_OBS` to `1` or `2`
- lowering `PRODUCT_LIMIT` to `2` or `3`

### 3) Search returns products but no usable 2D image HDU
Some JWST observations are spectroscopy-heavy or packaged differently. Try another target or filter to imaging observations only.

### 4) Cloud-backed download fails
The notebook will try to fall back to normal MAST downloads automatically.

### 5) You want a pure analysis workflow later from your own FITS files
You can reuse the later cells and replace the MAST download section with local file paths.
